# Factify 2 — Exploratory Data Analysis

Factify 2 is a **multi-modal fact verification** dataset from AAAI 2022.  
Each record pairs a **claim** (text + image URL) with a **reference document** (text + image URL)  
and asks: what is the relationship between the claim and the document?

**5-way label taxonomy:**
| Label | Meaning |
|---|---|
| `Support_Multimodal` | Both text AND image support the claim |
| `Support_Text` | Text supports, image is irrelevant or absent |
| `Insufficient_Multimodal` | Neither modality provides enough evidence |
| `Insufficient_Text` | Text alone is insufficient |
| `Refute` | Document contradicts the claim |

**Why this matters for our pipeline:** The `document` column is pre-fetched evidence — we can inject it directly as the evidence block, bypassing the live search node. This makes benchmarking essentially free.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')

TRAIN_PATH = Path('../datasets/Factify2/Factify 2/factify2_train/factify2/train.csv')
VAL_PATH   = Path('../datasets/Factify2/Factify 2/factify2_train/factify2/val.csv')
TEST_PATH  = Path('../datasets/Factify2/Factify 2/factify2_test/test.csv')

def load(path, split):
    df = pd.read_csv(path, sep='\t', engine='python', on_bad_lines='skip')
    df['split'] = split
    return df

train = load(TRAIN_PATH, 'train')
val   = load(VAL_PATH,   'val')
test  = load(TEST_PATH,  'test')

print(f'Train : {len(train):,} rows')
print(f'Val   : {len(val):,} rows')
print(f'Test  : {len(test):,} rows')
print(f'Columns: {list(train.columns)}')
train.head(2)

## 1. Label Distribution

In [ ]:
LABEL_ORDER  = ['Support_Multimodal', 'Support_Text', 'Insufficient_Multimodal', 'Insufficient_Text', 'Refute']
LABEL_COLORS = ['#1a9850', '#91cf60', '#fee08b', '#fc8d59', '#d73027']
LABEL_SHORT  = ['Sup\nMulti', 'Sup\nText', 'Insuf\nMulti', 'Insuf\nText', 'Refute']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, df) in zip(axes, [('Train', train), ('Val', val), ('Test', test)]):
    if 'Category' in df.columns:
        counts = df['Category'].value_counts().reindex(LABEL_ORDER, fill_value=0)
    else:
        counts = pd.Series([0]*5, index=LABEL_ORDER)
    bars = ax.bar(LABEL_SHORT, counts.values, color=LABEL_COLORS, edgecolor='white')
    ax.set_title(f'{name} (n={len(df):,})')
    ax.set_ylabel('Count')
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 20, str(v), ha='center', fontsize=8)

plt.suptitle('Label Distribution per Split', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print('\nTrain label counts:')
print(train['Category'].value_counts().reindex(LABEL_ORDER))

## 2. Mapping to Pipeline Verdict Schema

Our pipeline produces 3-way verdicts. Here's how Factify 2's 5 labels collapse:

In [ ]:
VERDICT_MAP = {
    'Support_Multimodal':      'supported',
    'Support_Text':            'supported',
    'Insufficient_Multimodal': 'misleading',
    'Insufficient_Text':       'misleading',
    'Refute':                  'refuted',
}
VERDICT_COLORS = {'supported': '#1a9850', 'misleading': '#fee08b', 'refuted': '#d73027'}

train['verdict'] = train['Category'].map(VERDICT_MAP)
val['verdict']   = val['Category'].map(VERDICT_MAP)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, df) in zip(axes, [('Train', train), ('Val', val)]):
    vc = df['verdict'].value_counts()
    colors = [VERDICT_COLORS[v] for v in vc.index]
    ax.pie(vc, labels=vc.index, autopct='%1.1f%%', colors=colors, startangle=90)
    ax.set_title(f'{name} — 3-way verdict mapping')

plt.tight_layout()
plt.show()
print('\nTrain verdict counts:')
print(train['verdict'].value_counts())

## 3. Text Length — Claim vs Document

In [ ]:
train['claim_words']    = train['claim'].fillna('').str.split().str.len()
train['doc_words']      = train['document'].fillna('').str.split().str.len()
train['claim_chars']    = train['claim'].fillna('').str.len()
train['doc_chars']      = train['document'].fillna('').str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Claim length
for label, color in zip(LABEL_ORDER, LABEL_COLORS):
    axes[0].hist(
        train[train['Category'] == label]['claim_words'].clip(upper=200),
        bins=40, alpha=0.5, label=label, color=color
    )
axes[0].set_title('Claim length (words)')
axes[0].set_xlabel('Words')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=7)

# Document length
for label, color in zip(LABEL_ORDER, LABEL_COLORS):
    axes[1].hist(
        train[train['Category'] == label]['doc_words'].clip(upper=1000),
        bins=40, alpha=0.5, label=label, color=color
    )
axes[1].set_title('Reference document length (words)')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Count')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

print('Median lengths by category:')
print(train.groupby('Category')[['claim_words', 'doc_words']].median().reindex(LABEL_ORDER).round(0))

## 4. Missing Data — Image URLs and Documents

In [ ]:
def is_present(s):
    return s.notna() & (s.str.strip() != '')

availability = pd.DataFrame({
    'claim_text':     is_present(train['claim']).groupby(train['Category']).mean(),
    'claim_image':    is_present(train['claim_image']).groupby(train['Category']).mean(),
    'document_text':  is_present(train['document']).groupby(train['Category']).mean(),
    'document_image': is_present(train['document_image']).groupby(train['Category']).mean(),
    'claim_ocr':      is_present(train['Claim OCR']).groupby(train['Category']).mean(),
    'doc_ocr':        is_present(train['Document OCR']).groupby(train['Category']).mean(),
}).reindex(LABEL_ORDER) * 100

availability.plot(kind='bar', figsize=(12, 5), colormap='Set2')
plt.title('Field availability % by label (train)')
plt.xlabel('Category')
plt.ylabel('% records with non-empty value')
plt.xticks(rotation=20)
plt.legend(bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

print('\nOverall availability (%):')
print(availability.mean().round(1).to_string())

## 5. Cross-modal structure — how often both modalities are present

In [ ]:
train['has_claim_img'] = is_present(train['claim_image'])
train['has_doc_img']   = is_present(train['document_image'])
train['has_doc_text']  = is_present(train['document'])
train['both_imgs']     = train['has_claim_img'] & train['has_doc_img']
train['full_multimodal'] = train['has_claim_img'] & train['has_doc_img'] & train['has_doc_text']

modal_stats = train.groupby('Category')[['has_claim_img', 'has_doc_img', 'has_doc_text', 'both_imgs', 'full_multimodal']].mean() * 100
modal_stats = modal_stats.reindex(LABEL_ORDER)

modal_stats.T.plot(kind='bar', figsize=(11, 5), color=LABEL_COLORS)
plt.title('Modality availability by category (%)')
plt.ylabel('%')
plt.xticks(rotation=15)
plt.legend(title='Category', bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.show()

print('\nFull multimodal (claim img + doc img + doc text) %:')
print(modal_stats['full_multimodal'].round(1))

## 6. Claim source — Twitter vs other

In [ ]:
train['claim_is_tweet'] = train['claim_image'].fillna('').str.contains('twimg|twitter', case=False)

tweet_pct = train.groupby('Category')['claim_is_tweet'].mean() * 100
tweet_pct = tweet_pct.reindex(LABEL_ORDER)

tweet_pct.plot(kind='bar', figsize=(8, 4), color=LABEL_COLORS, edgecolor='white')
plt.title('% of claims with Twitter image source')
plt.ylabel('%')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()
print(f'Overall claims from Twitter: {train["claim_is_tweet"].mean()*100:.1f}%')

## 7. Document length vs verdict — does more evidence help?

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for label, color in zip(LABEL_ORDER, LABEL_COLORS):
    subset = train[train['Category'] == label]['doc_words'].clip(upper=800)
    ax.boxplot(subset.dropna(), positions=[LABEL_ORDER.index(label)],
               patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.7),
               medianprops=dict(color='black', linewidth=2),
               widths=0.5)

ax.set_xticks(range(len(LABEL_ORDER)))
ax.set_xticklabels(LABEL_SHORT)
ax.set_title('Reference document length (words) by category')
ax.set_ylabel('Word count (clipped at 800)')
plt.tight_layout()
plt.show()

## 8. Benchmark strategy for our pipeline

### Option A — Free eval (inject document directly)
Skip the live search node entirely. Inject `document` as the evidence block.  
**Cost: $0 Tavily. Only OpenAI verdict synthesis.**

```python
# In the eval loop, put document text directly into retrieved_chunks
state = graph.invoke({
    "input": fact_check_input,
    "retrieved_chunks": [f"[REFERENCE DOCUMENT]\n{record.document[:2000]}"],
    "route": "factify2",   # skip live_search node
})
```

### Option B — Live search eval (standard pipeline)
Run the full pipeline with Tavily. Measures real-world performance.  
**Cost: ~$0.001 per record × 35,000 = ~$35 for full train set.**

### Recommended: start with Option A on val set (7,500 records)

In [ ]:
print('=== Val set summary ===')
print(f'Total records: {len(val):,}')
print(f'Label distribution:')
print(val['Category'].value_counts().reindex(LABEL_ORDER))
print(f'\nRecords with document text: {is_present(val["document"]).sum():,} ({is_present(val["document"]).mean()*100:.1f}%)')
print(f'Records with claim image:   {is_present(val["claim_image"]).sum():,} ({is_present(val["claim_image"]).mean()*100:.1f}%)')
print(f'Records with doc image:     {is_present(val["document_image"]).sum():,} ({is_present(val["document_image"]).mean()*100:.1f}%)')
print(f'\nMedian document length: {val["document"].fillna("").str.split().str.len().median():.0f} words')
print(f'Median claim length:    {val["claim"].fillna("").str.split().str.len().median():.0f} words')

## 9. Sample records per category

In [ ]:
for label in LABEL_ORDER:
    row = train[train['Category'] == label].iloc[0]
    print(f'\n{"="*60}')
    print(f'Category : {label}')
    print(f'Claim    : {str(row["claim"])[:200]}')
    print(f'Document : {str(row["document"])[:200]}')
    print(f'ClaimImg : {str(row["claim_image"])[:80]}')